> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 10 · Evaluations: an LLM judge for every answer

**CampusMate v7 (agent with a report card)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Observability tells you *what happened*. Evaluations tell you *how good it was*. You can't read 2,000 chats a day, so a second model, the **judge**, scores them against clear criteria. It's internal assessment for your agent, every day.

### What you'll do
- Learn evaluators, LLM-as-a-judge, and continuous vs spot checks
- Switch on online evaluation with one CLI command
- Spot-check one conversation in seconds
- Read the continuous scores

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Vocabulary

- **Evaluator**: one scoring rule, e.g. *Helpfulness*, *Goal success rate*, *Tool selection accuracy*.
- **LLM-as-a-judge**: a model reads the conversation (from the traces) and gives a score and an explanation.
- **Built-in evaluators** are named `Builtin.*`; you can also write your own.

| | Online (continuous) | Spot check (on demand) |
|---|---|---|
| When | Automatically, on live conversations | When you ask |
| Speed | Some minutes after a conversation goes quiet | Seconds |
| Good for | Watching quality over time | Checking one chat |

Each evaluation is itself a model call, so it costs a little. Real systems score a sample (e.g. 10%); our class scores 100%.

## Step 1: switch on online evaluation

(Terminal: `agentcore add online-eval --name qualityCheck --agent agent --evaluators Builtin.GoalSuccessRate Builtin.Helpfulness Builtin.ToolSelectionAccuracy --sampling-rate 100`, then `agentcore deploy`.)

In [ ]:
if "qualityCheck" in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text():
    print("qualityCheck is already in your project")
else:
    agentcore("add", "online-eval", "--name", "qualityCheck", "--agent", "agent", "--sampling-rate", "100",
              "--evaluators", "Builtin.GoalSuccessRate", "Builtin.Helpfulness", "Builtin.ToolSelectionAccuracy")
agentcore("deploy")

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## 🔬 Live check: spot-check one conversation right now

We make a new conversation, collect its traces from CloudWatch, and hand them to the judge.

In [ ]:
logs = boto3.client("logs", region_name=REGION)
judge = boto3.client("bedrock-agentcore", region_name=REGION)
session = f"lab10-{uuid.uuid4()}"
print(ask("Which semester is Infrastructure as Code in, and how is it assessed?", session))

def traces_for(session, tries=12):
    for _ in range(tries):
        records, token = [], None
        while True:
            kwargs = dict(logGroupName=LOG_GROUP, filterPattern=f'"{session}"', startTime=int((time.time() - 3600) * 1000))
            page = logs.filter_log_events(**({**kwargs, "nextToken": token} if token else kwargs))
            records += [json.loads(e["message"]) for e in page["events"] if e["message"].startswith("{")]
            token = page.get("nextToken")
            if not token:
                break
        if any(r.get("name", "").startswith("invoke_agent") for r in records):
            return records
        print("  waiting for the traces..."); time.sleep(10)
    return records

records = traces_for(session)
for evaluator in ("Builtin.Helpfulness", "Builtin.GoalSuccessRate"):
    result = judge.evaluate(evaluatorId=evaluator, evaluationInput={"sessionSpans": records})["evaluationResults"][0]
    print(f"\n{evaluator}: {result.get('value')} ({result.get('label')})\n  judge says: {result.get('explanation', '')[:300]}")

## Step 2: read the continuous scores

Online scores appear some minutes after conversations go quiet, so the chats from notebooks 08 and 09 may already be scored. If it's empty, come back after the break: that's normal.

In [ ]:
eval_id = resources["qualityCheck"]["arn"].split("/")[-1]
group = f"/aws/bedrock-agentcore/evaluations/results/{eval_id}"
try:     # CloudWatch can return empty pages while it searches, so we go through all pages
    pages = logs.get_paginator("filter_log_events").paginate(logGroupName=group, PaginationConfig={"MaxItems": 30})
    events = [e for page in pages for e in page["events"]]
except logs.exceptions.ResourceNotFoundException:
    events = []
print(f"{len(events)} scores so far")
for e in events[:15]:
    a = json.loads(e["message"]).get("attributes", {})
    print(f" - {a.get('gen_ai.evaluation.name', '?'):32s} {a.get('gen_ai.evaluation.score.value', '?'):>5} "
          f"({a.get('gen_ai.evaluation.score.label', '')})  session {str(a.get('session.id', ''))[:20]}")

### Reading scores like an examiner

- Low **ToolSelectionAccuracy** points at tool descriptions (remember Lab 9's hackathon question).
- Low **GoalSuccessRate** with high Helpfulness means polite, but the job wasn't done.
- Always read the **explanation**, not just the number.
- Surprise from our test runs: when CampusMate correctly **refused** a trick request (Lab 11), `GoalSuccessRate` scored it **0**.
  The judge grades the *user's* goal, not your rules.

## ✍️ Your turn

Spot-check your Lab 9 bad-answer conversation with `Builtin.ToolSelectionAccuracy`.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
BAD_SESSION = "TODO"   # TODO: paste the session ID from notebook 09 (starts with lab09-bad-)
if BAD_SESSION != "TODO":
    r = judge.evaluate(evaluatorId="Builtin.ToolSelectionAccuracy", evaluationInput={"sessionSpans": traces_for(BAD_SESSION)})
    print(r["evaluationResults"][0])
else:
    print("Skipped: set BAD_SESSION first.")

## ✅ Checkpoint

You are on track if:
- The deploy added `qualityCheck`
- The spot check printed scores between 0 and 1 with an explanation

**Stuck or behind?** Copy from `scripts/10_evaluations.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| 0 scores | Conversations are scored some minutes after they go quiet | Wait 15-30 minutes; use the spot check meanwhile |
| `evaluate` says no spans | The traces for a brand-new conversation haven't arrived | Re-run the spot-check cell after a minute |